# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [18]:
df = pd.read_csv('AviationData.csv',encoding='latin-1', low_memory=False)
state_codes = pd.read_csv('USState_Codes.csv')

print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
df.head(3)


print("Data Types")
print(df.dtypes)
print("\n Missing Values (%)")
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(missing[missing > 0].round(2))

df.describe()

Shape: (88889, 31)

Columns:
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date']
Data Types
Event.Id                   object
Investigation.Type         object
Accident.Number            object
Event.Date                 object
Location                   object
Country                    object
Latitude                   object
Longitude                  object
Airport.Code               object
Airport.Name               object
Injury.Severity            object
Aircraft.damage            object
Aircraft.

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [19]:
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df['Year'] = df['Event.Date'].dt.year

print(f"Records before filtering: {len(df)}")


df = df[(df['Year'] >= 1983) & (df['Year'] <= 2023)]
print(f" After year filter (1983-2023): {len(df)}")


df = df[df['Amateur.Built'].str.upper().isin(['NO', 'N'])]
print(f"After amateur-built filter: {len(df)}")

valid_categories = ['Airplane']
if 'Aircraft.Category' in df.columns:
    df = df[df['Aircraft.Category'].isin(valid_categories)]
    print(f"After airplane category filter: {len(df)}")

relevant_cols = ['Event.Date', 'Year', 'Make', 'Model', 'Number.of.Engines','Total.Fatal.Injuries', 'Total.Serious.Injuries','Total.Minor.Injuries',
                 'Total.Uninjured','Aircraft.damage', 'Injury.Severity', 'Weather.Condition', 'Broad.phase.of.flight', 'Engine.Type', 'Purpose.of.flight']

existing_cols = [c for c in relevant_cols if c in df.columns]
df = df[existing_cols + [c for c in df.columns if c not in existing_cols]]

print(f"\nFinal filtered shape: {df.shape}")

Records before filtering: 88889
 After year filter (1983-2023): 85289
After amateur-built filter: 76960
After airplane category filter: 21447

Final filtered shape: (21447, 32)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [20]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries','Total.Minor.Injuries', 'Total.Uninjured']

for col in injury_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)


df['Total.Aboard'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries'] + df['Total.Minor.Injuries'] + df['Total.Uninjured'])


df = df[df['Total.Aboard'] > 0]
print(f"After removing zero-aboard records: {len(df)}")


df['Serious.Injury.Rate'] = (
    (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Aboard']
).clip(0, 1)  

print('Serious.Injury.Rate sample stats:')
print(df['Serious.Injury.Rate'].describe().round(4))

After removing zero-aboard records: 20543
Serious.Injury.Rate sample stats:
count    20543.0000
mean         0.2840
std          0.4317
min          0.0000
25%          0.0000
50%          0.0000
75%          0.8000
max          1.0000
Name: Serious.Injury.Rate, dtype: float64


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [21]:
if 'Aircraft.damage' in df.columns:
    print('Aircraft.damage unique values:')
    print(df['Aircraft.damage'].value_counts(dropna=False))
    
    
    df['Aircraft.damage'] = df['Aircraft.damage'].str.strip().str.title().fillna('Unknown')
    
    
    df['Aircraft.Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)
    
    print("\nAircraft.Destroyed value counts: ")
    print(df['Aircraft.Destroyed'].value_counts())

Aircraft.damage unique values:
Aircraft.damage
Substantial    16821
Destroyed       2268
NaN              787
Minor            593
Unknown           74
Name: count, dtype: int64

Aircraft.Destroyed value counts: 
Aircraft.Destroyed
0    18275
1     2268
Name: count, dtype: int64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [22]:
print('Make Column Investigation')
print(f'Unique Makes before cleaning: {df["Make"].nunique()}')
print("\nTop 20 Makes: ")
print(df['Make'].value_counts().head(20))


"""
Make Column Cleaning Tasks:
1. Strip whitespace
2. Standardise to title case
3. Consolidate known variants (e.g. 'CESSNA', 'Cessna ', 'cessna' -> 'Cessna')
4. Replace NaN with 'Unknown'
5. For analysis: retain top N makes; group rest as 'Other'
"""

df['Make'] = df['Make'].str.strip().str.title().fillna('Unknown')


top_makes = df['Make'].value_counts().nlargest(50).index
df['Make.Group'] = df['Make'].apply(lambda x: x if x in top_makes else 'Other')

print(f'Unique Makes after cleaning: {df["Make"].nunique()}')
print(f'Make groups (incl Other): {df["Make.Group"].nunique()}')

Make Column Investigation
Unique Makes before cleaning: 1309

Top 20 Makes: 
Make
CESSNA                4794
PIPER                 2777
Cessna                2270
Piper                 1183
BEECH                 1003
BOEING                 642
Beech                  409
MOONEY                 235
CIRRUS DESIGN CORP     218
AIR TRACTOR INC        217
Boeing                 177
BELLANCA               158
AERONCA                149
MAULE                  144
Mooney                 125
Air Tractor            117
AIRBUS                 110
EMBRAER                104
LUSCOMBE                95
STINSON                 91
Name: count, dtype: int64
Unique Makes after cleaning: 1067
Make groups (incl Other): 51


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [23]:
print(f'NaNs in Model before: {df["Model"].isna().sum()}')
df['Model'] = df['Model'].fillna('Unknown')
print(f'NaNs in Model after:  {df["Model"].isna().sum()}')


model_make_counts = df.groupby(['Make', 'Model']).size().reset_index(name='Count').sort_values('Count', ascending=False)
print('Top 20 Make/Model combinations:')
print(model_make_counts.head(20).to_string(index=False))

model_make_check = df.groupby('Model')['Make'].nunique()
shared_models = model_make_check[model_make_check > 1].sort_values(ascending=False)

print(f"Models shared across multiple makes: {len(shared_models)}")
print('\nTop 10 shared model labels: ')
print(shared_models.head(10))
print('\nModel labels are NOT unique per make. A combined Make_Model key is needed.')

df['Make']  = df['Make'].str.strip().str.title().fillna('Unknown')
df['Model'] = df['Model'].str.strip().str.upper().fillna('Unknown')

df['Make_Model_ID'] = df['Make'].str.replace(' ', '_') + '__' + df['Model'].str.replace(' ', '_')

print(f'Unique Make_Model_IDs: {df["Make_Model_ID"].nunique()}')
print('\nSample Make_Model_IDs:')
print(df['Make_Model_ID'].value_counts().head(10))

NaNs in Model before: 15
NaNs in Model after:  0
Top 20 Make/Model combinations:
              Make     Model  Count
            Cessna       172    757
            Cessna       152    312
            Cessna       182    301
            Cessna      172S    272
             Piper      PA28    267
            Cessna      172N    246
            Cessna       180    213
            Boeing       737    199
            Cessna      172M    179
            Cessna       150    176
             Piper PA-18-150    175
             Piper PA-28-140    169
             Beech       A36    164
Cirrus Design Corp      SR22    144
            Cessna      172P    142
            Cessna       140    116
            Cessna      170B    107
            Cessna      172R    107
             Piper PA-28-180    105
             Piper PA-28-161    102
Models shared across multiple makes: 512

Top 10 shared model labels: 
Model
CHALLENGER II    11
G-164A           10
G-164B           10
S2R              10
A36   

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [24]:

cols_to_clean = ['Engine.Type', 'Weather.Condition', 'Number.of.Engines','Purpose.of.flight', 'Broad.phase.of.flight']

for col in cols_to_clean:
    if col in df.columns:
        n_null = df[col].isna().sum()
        pct_null = n_null / len(df) * 100
        n_unique = df[col].nunique()
        print(f"\n {col}")
        print(f"  NaNs: {n_null} ({pct_null:.1f}%)")
        print(f"  Unique values: {n_unique}")
        print(f"  Value counts:\n{df[col].value_counts(dropna=False).head(8).to_string()}")
        

if 'Engine.Type' in df.columns:
    df['Engine.Type'] = df['Engine.Type'].str.strip().str.title().fillna('Unknown')
    print('Engine.Type cleaned.')
    print(df['Engine.Type'].value_counts())
    
    

if 'Weather.Condition' in df.columns:
    df['Weather.Condition'] = df['Weather.Condition'].str.strip().str.upper()
    valid_wx = ['IMC', 'VMC']
    df['Weather.Condition'] = df['Weather.Condition'].apply(
        lambda x: x if x in valid_wx else 'Unknown'
    )
    print('Weather.Condition cleaned.')
    print(df['Weather.Condition'].value_counts())
    


if 'Number.of.Engines' in df.columns:
    df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce')
    median_engines = df['Number.of.Engines'].median()
    df['Number.of.Engines'] = df['Number.of.Engines'].fillna(median_engines)
    print(f'Number.of.Engines cleaned. Median imputed: {median_engines}')
    print(df['Number.of.Engines'].value_counts().sort_index())
    

if 'Purpose.of.flight' in df.columns:
    df['Purpose.of.flight'] = df['Purpose.of.flight'].str.strip().str.title().fillna('Unknown')
    print('Purpose.of.flight cleaned.')
    print(df['Purpose.of.flight'].value_counts().head(10))
    

if 'Broad.phase.of.flight' in df.columns:
    df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.strip().str.title().fillna('Unknown')
    print('Broad.phase.of.flight cleaned.')
    print(df['Broad.phase.of.flight'].value_counts().head(10))


 Engine.Type
  NaNs: 3295 (16.0%)
  Unique values: 9
  Value counts:
Engine.Type
Reciprocating    14986
NaN               3295
Turbo Prop        1225
Turbo Fan          797
Turbo Jet          127
Unknown            100
Turbo Shaft         10
Electric             1

 Weather.Condition
  NaNs: 2199 (10.7%)
  Unique values: 4
  Value counts:
Weather.Condition
VMC    17028
NaN     2199
IMC     1058
Unk      192
UNK       66

 Number.of.Engines
  NaNs: 1956 (9.5%)
  Unique values: 5
  Value counts:
Number.of.Engines
1.0    15689
2.0     2784
NaN     1956
4.0       69
3.0       37
0.0        8

 Purpose.of.flight
  NaNs: 2860 (13.9%)
  Unique values: 25
  Value counts:
Purpose.of.flight
Personal              11697
NaN                    2860
Instructional          2747
Aerial Application      933
Business                495
Positioning             350
Unknown                 335
Skydiving               164

 Broad.phase.of.flight
  NaNs: 17718 (86.2%)
  Unique values: 12
  Value counts:
Bro

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [25]:
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print('Missing % per column:')
print(missing_pct.round(2).to_string())


MISSING_THRESHOLD = 70.0

cols_to_drop = missing_pct[missing_pct > MISSING_THRESHOLD].index.tolist()

print(f"Columns to drop ( >{MISSING_THRESHOLD}% missing): {len(cols_to_drop)}")
print(cols_to_drop)

df.drop(columns=cols_to_drop, inplace=True)
print(f"\nShape after column removal: {df.shape}")


remaining_missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
remaining_missing = remaining_missing[remaining_missing > 0]
print('Remaining columns with missing values (%): ')
print(remaining_missing.round(2).to_string())

Missing % per column:
Schedule                  89.68
Air.carrier               52.84
Airport.Code              33.03
Airport.Name              32.64
Report.Status             19.03
Longitude                  7.79
Latitude                   7.76
Publication.Date           3.53
FAR.Description            1.60
Registration.Number        0.79
Location                   0.02
Country                    0.00
Total.Aboard               0.00
Amateur.Built              0.00
Aircraft.Category          0.00
Serious.Injury.Rate        0.00
Aircraft.Destroyed         0.00
Make.Group                 0.00
Event.Date                 0.00
Year                       0.00
Accident.Number            0.00
Make                       0.00
Model                      0.00
Number.of.Engines          0.00
Total.Fatal.Injuries       0.00
Total.Serious.Injuries     0.00
Total.Minor.Injuries       0.00
Total.Uninjured            0.00
Aircraft.damage            0.00
Injury.Severity            0.00
Weather.Condition 

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [26]:
"""
Reasons why CSV is saved here:
- Saves the cleaned/intermediate state to file so the analysis notebook can load it directly without re-running all cleaning steps.
- Keeps notebooks modular and readable: one notebook cleans, one analyses.
- Ensures reproducibility the exact cleaned dataset is preserved.
"""

output_path = 'AviationData_Cleaned.csv'
df.to_csv(output_path, index=False)

print(f" Cleaned data saved to: {output_path}")
print(f" Final shape: {df.shape}")
print(f" Columns: {df.columns.tolist()}")

 Cleaned data saved to: AviationData_Cleaned.csv
 Final shape: (20543, 36)
 Columns: ['Event.Date', 'Year', 'Make', 'Model', 'Number.of.Engines', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Aircraft.damage', 'Injury.Severity', 'Weather.Condition', 'Broad.phase.of.flight', 'Engine.Type', 'Purpose.of.flight', 'Event.Id', 'Investigation.Type', 'Accident.Number', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Aircraft.Category', 'Registration.Number', 'Amateur.Built', 'FAR.Description', 'Air.carrier', 'Report.Status', 'Publication.Date', 'Total.Aboard', 'Serious.Injury.Rate', 'Aircraft.Destroyed', 'Make.Group', 'Make_Model_ID']
